# Lab 05 — Spark Structured Streaming
**Lifecycle, micro-batch, event time, window, watermark, checkpoint, Kafka pattern**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
Memahami lifecycle query, micro-batch, trigger, progress, event time, window, watermark, checkpoint, dan pola integrasi Kafka. Semua query yang dijalankan di notebook harus dihentikan dengan `q.stop()`.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

## Helper — hentikan streaming query yang tertinggal

In [ ]:
for q in spark.streams.active:
    print("Stopping:", q.name, q.id)
    q.stop()
print("Active queries:", len(spark.streams.active))

## Lab 1 — Streaming tanpa Kafka: lifecycle + progress (15–20 menit)
Buat rate source 20 rows/s, `bucket=value%3`, groupBy bucket, output ke memory sink agar notebook mudah diamati.

In [ ]:
rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
)

bucketed = rate_df.withColumn("bucket", F.col("value") % 3)
counts = bucketed.groupBy("bucket").count()

print("isStreaming sebelum start:", counts.isStreaming)

In [ ]:
q1 = (
    counts.writeStream
    .format("memory")
    .queryName("lab_rate_counts")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

time.sleep(8)
spark.sql("SELECT * FROM lab_rate_counts ORDER BY bucket").show()
print(json.dumps(q1.lastProgress, indent=2)[:5000])
q1.stop()

**Cari:** `inputRowsPerSecond`, `processedRowsPerSecond`, `durationMs`. Jelaskan mengapa mendefinisikan DataFrame belum memproses stream sebelum `start()`.

## Lab 2 — Event time, window, watermark (25–30 menit)
Rate source menyediakan timestamp. Kita jadikan timestamp itu sebagai event time, buat region dan amount sintetis, lalu agregasi per window.

In [ ]:
events = (
    spark.readStream.format("rate").option("rowsPerSecond", 30).load()
    .select(
        F.col("timestamp").alias("event_time"),
        F.when((F.col("value") % 3) == 0, "JKT")
         .when((F.col("value") % 3) == 1, "BDG")
         .otherwise("SBY").alias("region"),
        ((F.col("value") % 100) * 10000 + 50000).cast("double").alias("amount"),
        "value"
    )
)

agg = (
    events
    .withWatermark("event_time", "10 seconds")
    .groupBy(F.window("event_time", "5 seconds"), "region")
    .agg(F.count("*").alias("n"), F.sum("amount").alias("total_amount"))
)

In [ ]:
q2 = (
    agg.writeStream
    .format("memory")
    .queryName("lab_window")
    .outputMode("update")
    .trigger(processingTime="2 seconds")
    .start()
)
time.sleep(10)
spark.sql("""
SELECT window.start, window.end, region, n, total_amount
FROM lab_window
ORDER BY window.start, region
""").show(50, truncate=False)
print(json.dumps(q2.lastProgress, indent=2)[:6000])
q2.stop()

**Diskusi:** mengapa window aggregation membutuhkan state? Apa peran watermark terhadap pertumbuhan state? Bedakan event time dan processing time.

## Lab 3 — File stream + checkpoint + restart (30 menit)
Lab ini lebih deterministik untuk menunjukkan checkpoint. Spark membaca JSON yang **ditambahkan setelah query dibuat**.

In [ ]:
stream_root = "/home/spark/datasets/streaming_lab"
input_dir = f"{stream_root}/input"
out_dir = f"{stream_root}/output"
ckpt_dir = f"{stream_root}/checkpoint"
for p in [stream_root]:
    if os.path.exists(p):
        shutil.rmtree(p)
os.makedirs(input_dir, exist_ok=True)

stream_schema = T.StructType([
    T.StructField("trx_id", T.StringType()),
    T.StructField("region", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("event_time", T.TimestampType()),
])

file_stream = spark.readStream.schema(stream_schema).json(input_dir)
valid_stream = file_stream.filter(F.col("trx_id").isNotNull() & (F.col("amount") >= 0))

In [ ]:
q3 = (
    valid_stream.writeStream
    .format("parquet")
    .option("path", out_dir)
    .option("checkpointLocation", ckpt_dir)
    .outputMode("append")
    .trigger(processingTime="2 seconds")
    .start()
)

batch1 = spark.createDataFrame([
    ("T1", "JKT", 100000.0, "2026-11-10 10:00:00"),
    ("T2", "BDG", 200000.0, "2026-11-10 10:00:01"),
], ["trx_id", "region", "amount", "event_time_raw"]).withColumn("event_time", F.to_timestamp("event_time_raw")).drop("event_time_raw")
batch1.coalesce(1).write.mode("append").json(input_dir)

time.sleep(6)
q3.stop()
print("output after batch1:", spark.read.parquet(out_dir).count())

In [ ]:
# Restart query dengan checkpoint SAMA, lalu tambahkan batch baru.
q3b = (
    valid_stream.writeStream
    .format("parquet")
    .option("path", out_dir)
    .option("checkpointLocation", ckpt_dir)
    .outputMode("append")
    .trigger(processingTime="2 seconds")
    .start()
)

batch2 = spark.createDataFrame([
    ("T3", "SBY", 300000.0, "2026-11-10 10:00:02"),
], ["trx_id", "region", "amount", "event_time_raw"]).withColumn("event_time", F.to_timestamp("event_time_raw")).drop("event_time_raw")
batch2.coalesce(1).write.mode("append").json(input_dir)

time.sleep(6)
q3b.stop()
print("output after restart + batch2:", spark.read.parquet(out_dir).count())

**Pertanyaan:** mengapa batch1 tidak diproses ulang saat query restart dengan checkpoint yang sama? Apa risiko jika checkpoint dihapus atau dipakai ulang untuk query berbeda?

## Lab 4 — Kafka preflight dan parsing schema (opsional, 30–40 menit)
Presentation menggunakan Kafka `kafka:9092`. Connector Spark–Kafka tidak selalu ada di image Spark. Jalankan preflight sebelum demo kelas.

In [ ]:
# Preflight connector. Ini tidak memulai streaming query.
try:
    _ = (spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", "kafka:9092")
         .option("subscribe", "transactions")
         .load())
    kafka_connector_available = True
except Exception as e:
    kafka_connector_available = False
    print("Kafka connector belum tersedia pada SparkSession/classpath.")
    print("Untuk training offline, bake package spark-sql-kafka-0-10_2.12:3.5.7 ke image.")
print("kafka_connector_available =", kafka_connector_available)

In [ ]:
kafka_schema = T.StructType([
    T.StructField("trx_id", T.StringType()),
    T.StructField("region", T.StringType()),
    T.StructField("amount", T.DecimalType(14,2)),
    T.StructField("event_time", T.TimestampType()),
])

if kafka_connector_available:
    raw = (spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", "transactions")
        .option("startingOffsets", "latest")
        .load())

    parsed = (
        raw.selectExpr(
            "CAST(value AS STRING) AS json",
            "topic", "partition", "offset", "timestamp"
        )
        .withColumn("data", F.from_json("json", kafka_schema))
        .select("data.*", "topic", "partition", "offset", "timestamp")
    )
    parsed.printSchema()

## Capstone — transaksi realtime
Rancang pipeline:
`Kafka/file source → parse → validate → watermark 10m → window 5m/region → sink`

**Quality:** `trx_id` null → quarantine, amount < 0 → invalid, simpan metadata offset jika Kafka.  
**Analytics:** count + total amount per event-time window/region.  
**Reliability:** checkpoint persisten, queryName, progress metrics, restart test.

Jelaskan juga mengapa “exactly-once Spark” tidak otomatis berarti persis satu efek pada external JDBC/API sink.

## Exit ticket
1. `readStream.load()` langsung memproses data atau belum?
2. Apa fungsi watermark?
3. Apa yang menunjukkan backlog?
4. Setelah checkpoint ada, apa peran `startingOffsets` saat restart?